# Determinación de la Edad Cinemática de Grupos Móviles Jóvenes con Gaia DR3

## Metodología

### Selección de los 3 Grupos Móviles (Con edad < 20 Myr, excepto LCC)

Para este estudio, lo más conveniente es escoger 3 asociaciones cercanas y bien pobladas en Simbad/Gaia, porque para calcular $(x, y, z, v_x, v_y, v_z)$ se requiere la velocidad radial, cuyo valor solo se reporta para cierta fracción de estrellas de Gaia DR3. Elegir un grupo pequeño o lejano puede implicar que, luego de filtrar, queden muy pocas estrellas.

Por esto, luego de estudiar la edad típica y la distancia media en la lista de Wikipedia [List of nearby stellar associations and moving groups](https://en.wikipedia.org/wiki/List_of_nearby_stellar_associations_and_moving_groups), los candidatos seleccionados fueron:

1. **TW Hydrae association (TWA):** Edad $\approx 8\text{--}10\text{ Myr}$, distancia media $\sim 55\text{ pc}$ ($\varpi \approx 18\text{ mas}$).
2. **$\eta$ Chamaeleontis cluster ($\eta$ Cha):** Edad $\approx 8\text{--}11\text{ Myr}$, distancia media $\sim 95\text{ pc}$ ($\varpi \approx 10.5\text{ mas}$).
3. **Upper Scorpius (USco):** Edad $\approx 10\text{--}11\text{ Myr}$, distancia media $\sim 145\text{ pc}$ ($\varpi \approx 6.9\text{ mas}$).

### Query (Simbad + Gaia TAP)

El flujo de adquisición de datos se estructura en dos pasos:
1. **Consulta a Simbad TAP (`simbad.harvard.edu`):** Mediante un `JOIN` entre `basic` e `ident` se recuperan los miembros catalogados de cada asociación junto con su identificador oficial `Gaia DR3 <source_id>`.
2. **Consulta espacial en Gaia DR3 (`gaiadr3.gaia_source`):** Mediante `CONTAINS(POINT('ICRS', ra, dec), CIRCLE(...))` y un filtro de paralaje se descargan todas las fuentes con velocidad radial medida (`radial_velocity IS NOT NULL`) en la región de cada grupo, incluyendo tanto los miembros confirmados como las estrellas vecinas del campo necesarias para la búsqueda posterior de nuevos candidatos mediante GMM.

In [ ]:
# Instalación de astroquery (descomentar si se ejecuta en un entorno nuevo como Google Colab)
# %pip install astroquery -Uq

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from astroquery.simbad import Simbad
from astroquery.gaia import Gaia
import astropy.units as u
from astropy.coordinates import SkyCoord

In [ ]:
# Configuración del servidor espejo de Simbad
my_simbad = Simbad()
my_simbad.server = "simbad.harvard.edu"

# Consultas ADQL para obtener los miembros catalogados en Simbad con ID de Gaia DR3
queries_simbad = {
    "twa": """
        SELECT DISTINCT basic.oid, basic.main_id, basic.ra, basic.dec, ident.id AS gaia_id
        FROM basic
        JOIN ident ON basic.oid = ident.oidref
        WHERE basic.oid IN (
            SELECT oidref FROM ident WHERE id LIKE 'TWA %'
        )
        AND ident.id LIKE 'Gaia DR3 %'
    """,
    "etacha": """
        SELECT DISTINCT basic.oid, basic.main_id, basic.ra, basic.dec, ident.id AS gaia_id
        FROM basic
        JOIN ident ON basic.oid = ident.oidref
        WHERE basic.oid IN (
            SELECT oidref FROM ident WHERE id LIKE 'RECX %' OR id LIKE 'eta Cha %'
        )
        AND ident.id LIKE 'Gaia DR3 %'
    """,
    "usco": """
        SELECT DISTINCT basic.oid, basic.main_id, basic.ra, basic.dec, ident.id AS gaia_id
        FROM basic
        JOIN ident ON basic.oid = ident.oidref
        WHERE basic.oid IN (
            SELECT oidref FROM ident WHERE id LIKE 'UScoCTIO %' OR id LIKE 'RX J16%'
        )
        AND ident.id LIKE 'Gaia DR3 %'
    """
}

for grupo, query in queries_simbad.items():
    archivo_simbad = f"simbad_{grupo}.csv"
    if os.path.exists(archivo_simbad):
        print(f"[{grupo.upper()}] Archivo local '{archivo_simbad}' ya existe. Saltando descarga.")
        continue
    print(f"[{grupo.upper()}] Consultando Simbad TAP...")
    tabla_simbad = my_simbad.query_tap(query)
    tabla_simbad.write(archivo_simbad, format="csv", overwrite=True)
    print(f"[{grupo.upper()}] Miembros en Simbad con Gaia DR3: {len(tabla_simbad)} -> '{archivo_simbad}'.")

In [ ]:
# Parámetros de los conos de búsqueda en Gaia DR3 (RA, Dec, radio en grados, rango de paralaje en mas)
parametros_gaia = {
    "twa":    {"ra": 165.0, "dec": -34.0, "radio": 25.0, "plx_min": 10.0, "plx_max": 30.0},
    "etacha": {"ra": 130.5, "dec": -79.0, "radio":  5.0, "plx_min":  7.0, "plx_max": 15.0},
    "usco":   {"ra": 243.0, "dec": -23.0, "radio":  8.0, "plx_min":  5.0, "plx_max": 10.0},
}

for grupo, p in parametros_gaia.items():
    archivo_gaia = f"gaia_{grupo}.csv"
    if os.path.exists(archivo_gaia):
        print(f"[{grupo.upper()}] Archivo local '{archivo_gaia}' ya existe. Saltando descarga.")
        continue

    query_gaia = f"""
    SELECT source_id, ra, dec, parallax, parallax_error,
           pmra, pmra_error, pmdec, pmdec_error,
           radial_velocity, radial_velocity_error, ruwe
    FROM gaiadr3.gaia_source
    WHERE 1 = CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {p['ra']}, {p['dec']}, {p['radio']})
    )
    AND parallax BETWEEN {p['plx_min']} AND {p['plx_max']}
    AND radial_velocity IS NOT NULL
    """

    print(f"[{grupo.upper()}] Lanzando consulta a Gaia DR3...")
    job = Gaia.launch_job_async(query_gaia)
    tabla_gaia = job.get_results()
    tabla_gaia.write(archivo_gaia, format="csv", overwrite=True)
    print(f"[{grupo.upper()}] Estrellas en Gaia con RV en la región: {len(tabla_gaia)} -> '{archivo_gaia}'.")

In [ ]:
# Cross-match entre Simbad y Gaia DR3 por source_id para los 3 grupos
grupos = ["twa", "etacha", "usco"]
datos_grupos = {}
resumen_cruce = []

for grupo in grupos:
    df_simbad = pd.read_csv(f"simbad_{grupo}.csv")
    df_gaia = pd.read_csv(f"gaia_{grupo}.csv")

    # Extraemos el número limpio de source_id quitando el texto 'Gaia DR3 '
    ids_simbad = (
        df_simbad["gaia_id"]
        .astype(str)
        .str.replace("Gaia DR3 ", "", regex=False)
        .str.strip()
        .astype("int64")
    )

    # Creamos la columna booleana de pertenencia confirmada según Simbad
    df_gaia["es_miembro_simbad"] = df_gaia["source_id"].isin(ids_simbad)
    datos_grupos[grupo] = df_gaia

    faltantes = set(ids_simbad) - set(df_gaia["source_id"])

    resumen_cruce.append({
        "Grupo": grupo.upper(),
        "Catalogados Simbad (con ID Gaia)": len(set(ids_simbad)),
        "Estrellas en cono Gaia (con RV)": len(df_gaia),
        "Miembros Simbad cruzados (con RV)": int(df_gaia["es_miembro_simbad"].sum()),
        "IDs Simbad fuera del cono/RV": len(faltantes)
    })

df_resumen = pd.DataFrame(resumen_cruce)
df_resumen